In [1]:
import numpy as np
from amuse.ic.kroupa import new_kroupa_mass_distribution
from amuse.units import units
from amuse.ic.salpeter import new_salpeter_mass_distribution
from amuse.lab import nbody_system
from amuse.lab import *
from amuse.ext.orbital_elements import new_binary_from_orbital_elements
from amuse.plot import scatter
from amuse.ext.orbital_elements import get_orbital_elements_from_binary
from amuse.community.ph4.interface import Ph4
from amuse.ext.orbital_elements import semimajor_axis_to_orbital_period
from amuse.units.core import named_unit
from amuse.units import units

## Derivations for parameter sweep:
Begin with full energy and angular momentum expressions (in COM frame)
### $E = \frac{1}{2}m_1v_1^2 + \frac{1}{2}m_2v_2^2 - \frac{Gm_1m_2}{|r_1 - r_2|}$
### $L = m_1(r_1 \times v_1) + m_2(r_2 \times v_2)$

also the COM frame implies:
### $m_1 r_1 + m_2 r_2 = 0$
### $m_1 v_1 + m_2 v_2 = 0$

reduction to one body:
### $r = r_1 - r_2$
### $v = v_1 - v_2$
### $M = m_1 + m_2$
### $\mu = \frac{m_1 m_2}{m_1+m_2}$
inserting which  into the COM conditions gives:
### $r_1 = \frac{m_2 r}{M}$, $r_2 = -\frac{m_1 r}{M}$
### $v_1 = \frac{m_2 v}{M}$, $v_2 = -\frac{m_1 v}{M}$
insert these into E and L expressions to obtain an equivalent one-body problem with mass $\mu$:
### $E = \frac{1}{2}\mu v^2 - \frac{Gm_1 m_2}{r}$ 
### $L = \mu (r \times v)$
now use Keplerian orbit formulation
### $v^2 = \dot{r}^2 + r^2\dot{\theta}^2$ 
### $L = \mu r^2 \dot{\theta} => \dot{\theta} = \frac{L}{\mu r^2}$
substitute $v^2$ into energy equation:
### $E = \frac{1}{2} \mu \dot{r}^2 + \frac{L^2}{2\mu r^2}-\frac{Gm_1 m_2}{r}$
at pericentre $\dot{r} = 0$ =>

# $E = \frac{L^2}{2\mu r_p^2}-\frac{Gm_1 m_2}{r_p}$

then substitute the orbit (conic section) equation
### $r(f) = \frac{p}{1+e \ cos f}$, where f is true anomaly
into the above energy equation to get the eccentricity:

# $e = \sqrt{1+\frac{2EL^2}{\mu^3 G^2 (m_1+m_2)^2}}$
again from the conic section equation, setting r=r_p and inserting eccentricity we get:
# $r_p = \frac{L^2}{\mu G m_1 m_2 (1+e)}$



so, for the sweep we need:
### $m_1, m_2, E, L$ => $e, r_p$

to initialize initial conditions, we again use the relative seperation:
### $r(f) = \frac{p}{1+e \ cos f}$, where f is true anomaly and p:
### $p = \frac{L^2}{\mu G (m_1 + m_2)}$

now, $r$ and $v$ vectors are expressed as:
### $r = r(f) \ [(cos f)_x \ (sin f)_y]$
### $v = \frac{G(m_1 + m_2) \mu}{L} \ [(-sin f)_x \ (e+cos f)_y]$

and from the previous expressions we already have:
### $r_1 = \frac{m_2 r}{M}$, $r_2 = -\frac{m_1 r}{M}$
### $v_1 = \frac{m_2 v}{M}$, $v_2 = -\frac{m_1 v}{M}$

to initialize one simulation we can choose an r value (that we keep the same for all cases)
from the r value we find the true anomaly f and thus the initial r & v values!

In [2]:
MBlackhole = named_unit("70 solar masses", "MBH70", 70 * units.MSun)

In [3]:
from amuse.units import units, constants 
import numpy as np
#These are hypothetical values for the system (will change accordingly)
m1 = 10 | units.MSun                    # original black hole
m2 = 1 | units.MSun                     # original companion star
m3 = 10 | units.MSun                    # incoming black hole
b = 1.5 | units.AU                       # impact parameter of the incoming black hole*                
v_inf = 20 | units.kms                     # relative velocity of the incoming black hole*

M12 = m1 + m2
M_total = M12 +m3
mu_out = (M12 * m3) / M_total

#Incoming orbit relative to the original binary's center of mass
E_out 0.5 * mu_out * v_inf**2
L_out = mu_out * b * v_inf

#Keplerian outer-orbit parameters
e_out = np.sqrt(1 + (2 * E_out * L_out**2) / (mu_out ** 3 *(constants.G * M_total)**2))
p_out = L_out**2 / (mu_out * constants.G * M_total)
r_peri = p_out / (1 + e_out)
print("Outer-orbit energy", E_out)
print("Outer angular momentum", L_out)
print("Outer-orbit eccentricity", e_out)
print("Pericenter:", r_peri)


SyntaxError: invalid syntax (1624161503.py, line 15)

In [4]:
def get_orb_params(m1, m2, E, L, initial_seperation):
    """given masses of the 2 objects (m2 being the 'incoming' object), total energy, total angular
    momentum and the initial seperation, calculates corresponding eccentricity and true anomaly"""
    
    #needed values for calculation (reduced mass & semi latus rectum)
    red_mass = m1*m2/(m1+m2)
    p = (L**2)/(red_mass*(m1+m2)*constants.G)

    #calculate eccentricity of the orbit
    eccentricity = np.sqrt(1+((2*E*L**2)/((red_mass**3)*((m1+m2)**2)*constants.G)))

    #given eccentricity and p (semi latus rectum) we can then get the true anomaly corresponding to the initial seperation
    #we need this true anomaly to get the initial conditions for the run
    true_anom = np.arccos((p-initial_seperation)/(initial_seperation*eccentricity))

    return eccentricity, true_anom


def get_initial_conditions(true_anom, eccentricity, m1, m2, L):
    """given the true anomaly and eccentricity of the orbit, and the two masses and total angular momentum
    calculates the initial conditions in the COM frame (v1, r1, v2, r2)"""

    #needed values for calculation (reduced mass, total mass & semi latus rectum)
    red_mass = m1*m2/(m1+m2)
    totmass = m1+m2
    p = (L**2)/(red_mass*totmass*constants.G)

    #COM v and r values
    r_com = p/(1+eccentricity*np.cos(true_anom))
    v_com = (constants.G*red_mass*totmass)/L

    r_com_x = r_com*np.cos(true_anom)
    r_com_y = r_com*np.sin(true_anom)
    v_com_x = v_com*(-np.sin(true_anom))
    v_com_y = v_com*(eccentricity+np.cos(true_anom))

    #now converting these to r1, v1, r2, v2
    r1_x = r_com_x*m2/totmass
    r1_y = r_com_y*m2/totmass
    r2_x = -r_com_x*m1/totmass
    r2_y = -r_com_y*m1/totmass

    v1_x = v_com_x*m2/totmass
    v1_y = v_com_y*m2/totmass
    v2_x = -v_com_x*m1/totmass
    v2_y = -v_com_y*m1/totmass

    return r1_x, r1_y, v1_x, v1_y, r2_x, r2_y, v2_x, v2_y

In [5]:
array_of_energies = np.random.rand(10) #size of 10 t0 start
array_of_angular_momenta = np.random.rand(10) #size of 10 to start

eccentricities = []
true_anomalies = []
for i in array_of_energies:
    for j in array_of_angular_momenta: 
        eccentricity, true_anom = get_orb_params((m1, m2, i, j, initial_seperation))
        eccentricities.append(eccentricity)
        true_anomalies.append(true_anom)
eccentricities = np.array(eccentricities)
true_anomalies = np.array(true_anomalies)
for idx in range(len(eccentricities)):
    i = eccentricities[idx]
    j = true_anomalies[idx]
    r1_x, r1_y, v1_x, v1_y, r2_x, r2_y, v2_x, v2_y = get_initial_conditions((m1, m2, i, j, initial_seperation))
        

NameError: name 'm1' is not defined

In [8]:
#for a 2 body system

def initial_conditions_blackhole_star_system():
    mass_blackhole_1 = 1 | MBlackhole
    mass_blackhole_2 = 90/70 | MBlackhole
    semimajor_axis_blackhole_2 = 300 | units.AU
    eccentricity = eccentricity
    pericenter = pericenter

    # star_blackhole_binary = new_binary_from_orbital_elements( #inner binary
    #     mass1=mass_blackhole_1,
    #     mass2=mass_star,
    #     semimajor_axis=semimajor_axis_star,
    #     eccentricity=eccentricity,
    #     G=constants.G
    # )
    # star_blackhole_binary.name = ['Black Hole 1', 'Star']

    blackhole_binary = new_binary_from_orbital_elements( #outer binary
        mass1 = mass_blackhole_1, 
        mass2 = mass_blackhole_2, 
        semimajor_axis = semimajor_axis_blackhole_2, 
        eccentricity = eccentricity, 
        pericenter = pericenter,
        G=constants.G 
    )
    blackhole_binary.name = ['Fake Black Hole 1 (do not use)', 'Black Hole 2']

    # Shift the inner binary (star and black hole 1) to match the position of the fake black hole 1 in the outer binary
    fake_black_hole_1 = blackhole_binary[blackhole_binary.name == 'Fake Black Hole 1 (do not use)']
    shift_pos = fake_black_hole_1.position - star_blackhole_binary[0].position
    shift_vel = fake_black_hole_1.velocity - star_blackhole_binary[0].velocity
    star_blackhole_binary.position += shift_pos
    star_blackhole_binary.velocity += shift_vel

    #position black hole 2 relative to the orter binary orbit
    blackhole_2 = blackhole_binary[blackhole_binary.name == 'Black Hole 2'].copy()
    blackhole_2.position = star_blackhole_binary[0].position + (blackhole_binary[1].position - blackhole_binary[0].position)
    blackhole_2.velocity = star_blackhole_binary[0].velocity + (blackhole_binary[1].velocity - blackhole_binary[0].velocity)
    blackhole_2.name = ['Black Hole 2']
    
    star_blackhole_binary.add_particles(blackhole_2)
    star_both_blackholes = star_blackhole_binary.copy()
    return star_both_blackholes

if __name__ == "__main__":
    for idx, e_out in enumerate(eccentricities):
        for jdx, r_peri in enumerate(pericenters):
            star_both_blackholes = initial_conditions_blackhole_star_system(
                eccentricity_out = e_out, 
                pericenter_out = r_peri,
                eccentricity_star=0.0, 
                semimajor_axis_star=30|units.AU
                )
            
            a = [] | units.au
            e = []
    

            orbit_inner = get_orbital_elements_from_binary(star_both_blackholes[0:2], G = constants.G)
            print(f'black hole 1 and star orbit: {orbit_inner}')
            a.append(orbit_inner[2])
            e.append(orbit_inner[3])


            combined_inner = Particles(1) #star and black hole 1 together
            combined_inner.mass = star_both_blackholes[0:2].mass.sum()
            combined_inner.position = star_both_blackholes[0:2].center_of_mass()
            combined_inner.velocity = star_both_blackholes[0:2].center_of_mass_velocity()

            outer_system = Particles(2) #combined inner binary and black hole 2
            outer_system.add_particle(combined_inner[0])
            outer_system.add_particle(star_both_blackholes[2])
            
            orbit_outer = get_orbital_elements_from_binary(outer_system, G = constants.G)
            print(f'black hole 1/star and black hole 2 orbit: {orbit_outer}')
            a.append(orbit_outer[2])
            e.append(orbit_outer[3])

            scatter(star_both_blackholes.x, star_both_blackholes.y, s=2+300*np.sqrt(star_both_blackholes.mass/star_both_blackholes.mass.sum()), c='b', marker="*")

            scatter(a, e)

            converter = nbody_system.nbody_to_si(star_both_blackholes.mass.sum(), star_both_blackholes[0].position.length())
            gravity = Ph4(converter)
            gravity.parameters.timestep_parameter = 0.01
            gravity.particles.add_particles(star_both_blackholes)
            print(f'gravity.particles: {gravity.particles}')
            print(f'bodies (star and bh1 and bh2): {star_both_blackholes}')

            channel_from_gravity = gravity.particles.new_channel_to(star_both_blackholes)
            channel_from_gravity.copy()
            print(f'updated bodies (star and bh1 and bh2): {star_both_blackholes}')

            Porb = semimajor_axis_to_orbital_period(a[0], star_both_blackholes[0:2].mass.sum())
            print(f'Orbital period of black hole 1 and star: {Porb}')

            Porb2 = semimajor_axis_to_orbital_period(a[1], star_both_blackholes[0:2].mass.sum() + star_both_blackholes[2].mass)
            print(f'Orbital period of inner binary and black hole 2: {Porb2}')

            Ek0 = star_both_blackholes.kinetic_energy()
            Ep0 = star_both_blackholes.potential_energy()
            Etot0 = Ek0 + Ep0

            time = 0 | units.yr
            dt = Porb / 100
            index = 0
            while gravity.model_time < Porb:
                time += dt
                gravity.evolve_model(time)
                channel_from_gravity.copy()

                Ek = star_both_blackholes.kinetic_energy()
                Ep = star_both_blackholes.potential_energy()
                Etot = Ek + Ep
                
                print(f"time={time.in_(units.yr)}, dE = {(Etot-Etot0)/Etot0}")

                plt.xlim(-500, 500)
                plt.ylim(-500, 500)
                scatter(star_both_blackholes.x, star_both_blackholes.y, s=2+300*np.sqrt(star_both_blackholes.mass/star_both_blackholes.mass.sum()))
                index += 1        
                filename = f"binary_snapshot_e{idx}_p{jdx}_i{index:04}.jpg"
                plt.savefig( filename)
                plt.close()
            gravity.stop()

In [31]:
eccentricities = [0.1, 0.3, 0.5]
semimajor_axes = [100, 200, 300 | units.AU]
e_out = 0.5
axes = 100 | units.AU


In [32]:
#this is for a 3-body system

def initial_conditions_blackhole_star_system(eccentricity_out, eccentricity_star=0.0, semimajor_axis_star=30|units.AU):
    mass_blackhole_1 = 1 | MBlackhole
    mass_blackhole_2 = 90/70 | MBlackhole
    mass_star = 1/70 | MBlackhole
    semimajor_axis_star = 30 | units.AU
    eccentricity_star = 0.0
    semimajor_axis_blackhole_2 = 300 | units.AU
    eccentricity = eccentricity_out

    star_blackhole_binary = new_binary_from_orbital_elements( #inner binary
        mass1=mass_blackhole_1,
        mass2=mass_star,
        semimajor_axis=semimajor_axis_star,
        eccentricity=eccentricity_star,
        G=constants.G
    )
    star_blackhole_binary.name = ['Black Hole 1', 'Star']

    inner_binary_mass = mass_blackhole_1 + mass_star
    total_mass = inner_binary_mass + mass_blackhole_2

    blackhole_binary = new_binary_from_orbital_elements( #outer binary
        mass1 = inner_binary_mass, 
        mass2 = mass_blackhole_2, 
        semimajor_axis = None, 
        eccentricity = eccentricity, 
        G=constants.G 
    )
    blackhole_binary.name = ['Fake Black Hole 1 (do not use)', 'Black Hole 2']

    # Shift the inner binary (star and black hole 1) center of mass to match the position of the fake black hole 1 in the outer binary
    fake_black_hole_1 = blackhole_binary[blackhole_binary.name == 'Fake Black Hole 1 (do not use)']
    com_pos = star_blackhole_binary.center_of_mass()
    com_vel = star_blackhole_binary.center_of_mass_velocity()
    shift_pos = fake_black_hole_1.position - com_pos
    shift_vel = fake_black_hole_1.velocity - com_vel
    star_blackhole_binary.position += shift_pos
    star_blackhole_binary.velocity += shift_vel

    #position black hole 2 relative to the orter binary orbit
    blackhole_2 = blackhole_binary[blackhole_binary.name == 'Black Hole 2'].copy()
    blackhole_2.position = star_blackhole_binary.center_of_mass() + (blackhole_binary[1].position - blackhole_binary[0].position)
    blackhole_2.velocity = star_blackhole_binary.center_of_mass_velocity() + (blackhole_binary[1].velocity - blackhole_binary[0].velocity)
    blackhole_2.name = ['Black Hole 2']
    
    star_blackhole_binary.add_particles(blackhole_2)
    star_both_blackholes = star_blackhole_binary.copy()
    return star_both_blackholes



In [ ]:
import sys

print("--- DIAGNOSTIC CHECK ---", flush=True)
print(f"eccentricities type: {type(eccentricities) if 'eccentricities' in globals() else 'NOT DEFINED'}", flush=True)
print(f"eccentricities length: {len(eccentricities) if 'eccentricities' in globals() else 'N/A'}", flush=True)
print(f"semimajor_axes type: {type(semimajor_axes) if 'semimajor_axes' in globals() else 'NOT DEFINED'}", flush=True)
print(f"semimajor_axes length: {len(semimajor_axes) if 'semimajor_axes' in globals() else 'N/A'}", flush=True)
print("------------------------\n", flush=True)

for idx, e_out in enumerate(eccentricities):
    for jdx, axis in enumerate(semimajor_axes):
        star_both_blackholes = initial_conditions_blackhole_star_system(
            eccentricity_out = e_out, 
            eccentricity_star=0.0, 
            semimajor_axis_star = axis
            )
        
        a = [] | units.au
        e = []

        orbit_inner = get_orbital_elements_from_binary(star_both_blackholes[0:2], G = constants.G)
        print(f'black hole 1 and star orbit: {orbit_inner}')
        a.append(orbit_inner[2])
        e.append(orbit_inner[3])

        combined_inner = Particles(1) #star and black hole 1 together
        combined_inner.mass = star_both_blackholes[0:2].mass.sum()
        combined_inner.position = star_both_blackholes[0:2].center_of_mass()
        combined_inner.velocity = star_both_blackholes[0:2].center_of_mass_velocity()

        outer_system = Particles(2) #combined inner binary and black hole 2
        outer_system.add_particle(combined_inner[0])
        outer_system.add_particle(star_both_blackholes[2])
        
        orbit_outer = get_orbital_elements_from_binary(outer_system, G = constants.G)
        print(f'black hole 1/star and black hole 2 orbit: {orbit_outer}')
        a.append(orbit_outer[2])
        e.append(orbit_outer[3])

        scatter(star_both_blackholes.x, star_both_blackholes.y, s=2+300*np.sqrt(star_both_blackholes.mass/star_both_blackholes.mass.sum()), c='b', marker="*")

        scatter(a, e)

        converter = nbody_system.nbody_to_si(star_both_blackholes.mass.sum(), star_both_blackholes[0].position.length())
        gravity = Ph4(converter)
        gravity.parameters.timestep_parameter = 0.01
        gravity.particles.add_particles(star_both_blackholes)
        print(f'gravity.particles: {gravity.particles}')
        print(f'bodies (star and bh1 and bh2): {star_both_blackholes}')

        channel_from_gravity = gravity.particles.new_channel_to(star_both_blackholes)
        channel_from_gravity.copy()
        print(f'updated bodies (star and bh1 and bh2): {star_both_blackholes}')

        Porb = semimajor_axis_to_orbital_period(a[0], star_both_blackholes[0:2].mass.sum())
        print(f'Orbital period of black hole 1 and star: {Porb}')

        Porb2 = semimajor_axis_to_orbital_period(a[1], star_both_blackholes[0:2].mass.sum() + star_both_blackholes[2].mass)
        print(f'Orbital period of inner binary and black hole 2: {Porb2}')

        Ek0 = star_both_blackholes.kinetic_energy()
        Ep0 = star_both_blackholes.potential_energy()
        Etot0 = Ek0 + Ep0

        time = 0 | units.yr
        dt = Porb / 100
        index = 0
        while gravity.model_time < Porb:
            time += dt
            gravity.evolve_model(time)
            channel_from_gravity.copy()

            Ek = star_both_blackholes.kinetic_energy()
            Ep = star_both_blackholes.potential_energy()
            Etot = Ek + Ep
            
            print(f"time={time.in_(units.yr)}, dE = {(Etot-Etot0)/Etot0}")

            plt.xlim(-500, 500)
            plt.ylim(-500, 500)
            scatter(star_both_blackholes.x, star_both_blackholes.y, s=2+300*np.sqrt(star_both_blackholes.mass/star_both_blackholes.mass.sum()))
            index += 1        
            filename = f"full_system_snapshot_e{idx}_p{jdx}_i{index:04}.jpg"
            plt.savefig( filename)
            plt.close()
        gravity.stop()

--- DIAGNOSTIC CHECK ---
eccentricities type: <class 'list'>
eccentricities length: 3
semimajor_axes type: <class 'list'>
semimajor_axes length: 3
------------------------



/Users/emorymurff/anaconda3/envs/amuse/lib/python3.14/site-packages/amuse/ext/orbital_elements.py:344: UserWarning: new_binary_from_orbital_elements is deprecated, use generate_binaries instead
  warnings.warn(


TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'

# unused code put in markdown cell

def initial_conditions_star_blackhole():
    # Define initial conditions for the black hole binary
    mass_blackhole_1 = 1 | MBlackhole
    mass_star = 1/70 | MBlackhole
    semimajor_axis_star = 30 | units.AU
    eccentricity_star = 0.0

    bodies = new_binary_from_orbital_elements(
        mass1 = mass_blackhole_1, 
        mass2 = mass_star, 
        semimajor_axis = semimajor_axis_star, 
        eccentricity = eccentricity_star, 
        G=constants.G) 
    bodies.name = ['Black Hole', 'Star']
    return bodies

def compute_energy(particles):
    return particles.kinetic_energy() + particles.potential_energy()

if __name__ == "__main__": #black hole and star binary system
    bodies = initial_conditions_star_blackhole()
    print(bodies)

    sma_star = []
    e_star = []

    star_orbit = get_orbital_elements_from_binary(bodies[0:2], constants.G)
    print(f'star orbit: {star_orbit}')

    sma_star.append(star_orbit[2])
    e_star.append(star_orbit[3])

    converter = nbody_system.nbody_to_si(bodies.mass.sum(), 30 | units.AU)
    gravity = Ph4(converter)
    gravity.particles.add_particles(bodies) #makes a copy of bodies in the nbody code called gravity
    channel_from_gravity_to_bodies = gravity.particles.new_channel_to(bodies) #makes a solid link from the memory in the python script to the memory in the nbody code

    Ek0 = bodies.kinetic_energy()
    Ep0 = bodies.potential_energy()
    Etot = Ek0 + Ep0

    Porb = semimajor_axis_to_orbital_period(sma_star[0], bodies[0:2].mass.sum())
    time = 0 | units.yr
    dt = Porb / 100
    #index = 0
    while gravity.model_time < Porb:
        time += dt
        gravity.evolve_model(time)
        channel_from_gravity_to_bodies.copy()
        Ek = bodies.kinetic_energy()
        Ep = bodies.potential_energy()
        #print(f'time = {time.in_(units.yr)}, dEtot = {(Ek+Ep-Etot)/Etot}')
        plt.xlim(-10, 10)
        plt.ylim(-10, 10)
        scatter(bodies.x, bodies.y, s = 2+300*np.sqrt(bodies.mass/bodies.mass.sum()), c = 'blue')
        #index += 1
        #filename = f'snapshot_i{index:04}.jpg' #makes images for the movie. has a lot of image outputs
        #plt.savefig(filename)
        #plt.close()




incoming_velocity_inf = 20 | units.kms #incoming velocity at infinty
impact_parameter = 1.5 | units.AU

energy = 1/2*incoming_velocity_inf**2
angular_momentum = incoming_velocity_inf * impact_parameter




mass_blackhole_1 = 1 | MBlackhole
mass_star = 1/70 | MBlackhole
mass_blackhole_2 = 90/70 | MBlackhole
Mtot = mass_star + mass_blackhole_1 + mass_blackhole_2

def eccentricity_and_pericenter(energy, angular_momentum, Mtot):
    eccentricity_blackhole = (1 + (2 * energy * angular_momentum**2)/((constants.G * Mtot)**2))**0.5
    pericenter_blackhole = angular_momentum**2 / ((constants.G * Mtot) * (1 + eccentricity_blackhole))
    return eccentricity_blackhole, pericenter_blackhole




eccentricity_blackhole_2, pericenter_blackhole_2 = eccentricity_and_pericenter(energy, angular_momentum, Mtot)
print(eccentricity_blackhole, pericenter_blackhole)





def initial_conditions_blackhole_binary():
    mass_blackhole_1 = 1 | MBlackhole
    mass_blackhole_2 = 90/70 | MBlackhole
    semimajor_axis_blackhole = 300 | units.AU
    eccentricity_blackhole = 0.0

    bodies = new_binary_from_orbital_elements(
        mass1 = mass_blackhole_1, 
        mass2 = mass_blackhole_2, 
        semimajor_axis = semimajor_axis_blackhole, 
        eccentricity = eccentricity_blackhole_2, 
        pericenter = pericenter_blackhole_2,
        G=constants.G) 
    bodies.name = ['Black Hole 1', 'Black Hole 2']
    return bodies